### CUSTOMERS TABLE

In [0]:
from pyspark.sql.functions import col, when, trim, coalesce, lit, row_number, monotonically_increasing_id, try_to_date
from pyspark.sql import Window

# Read from bronze
df = spark.table("novocart.bronze.customers_raw")

# 1. Deduplicate using business key
window_spec = Window.partitionBy("customer_id").orderBy(monotonically_increasing_id())

df_clean = df.withColumn("row_num", row_number().over(window_spec)) \
             .filter(col("row_num") == 1) \
             .drop("row_num")

# 2. Clean customer_name 
df_clean = df_clean.withColumn(
    "customer_name",
    when(col("customer_name").isNull(), "Unknown Customer")
    .when(col("customer_name").isin("NULL", "\\N", "?"), "Unknown Customer")
    .otherwise(trim(col("customer_name")))
)

# 3. Clean email 
df_clean = df_clean.withColumn(
    "email",
    when(col("email").isNull(), "noemail@unknown.com")
    .when(col("email").isin("NULL", "\\N", "?"), "noemail@unknown.com")
    .otherwise(trim(col("email")))
)

# 4. Standardize date format , handle nulls
df_clean = df_clean.withColumn(
    "registration_date",
    coalesce(
        try_to_date(col("registration_date"), "yyyy-MM-dd"),
        try_to_date(col("registration_date"), "yyyy/MM/dd"),
        try_to_date(col("registration_date"), "dd/MM/yyyy")
    )
)

#  replace null dates 
df_clean = df_clean.withColumn(
    "registration_date",
    coalesce(col("registration_date"), lit("1900-01-01").cast("date"))
)

# 5. Clean other columns
df_clean = df_clean.withColumn("country_code", trim(col("country_code"))) \
                   .withColumn("channel", trim(col("channel")))

# 6. Final selection
df_silver = df_clean.select(
    "customer_id",
    "customer_name",
    "email",
    "registration_date",
    "country_code",
    "channel"
)

# 7. Write to silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novocart.silver.customers_cleaned")

# 8. Display
display(spark.table("novocart.silver.customers_cleaned"))

### ORDERS TABLE

In [0]:
from pyspark.sql.functions import col, when, trim, coalesce, lit, row_number, monotonically_increasing_id, try_to_date
from pyspark.sql import Window

# Read from bronze
df = spark.table("novocart.bronze.orders_raw")

# 1. Deduplicate using business key
window_spec = Window.partitionBy("order_id").orderBy(monotonically_increasing_id())

df_clean = df.withColumn("row_num", row_number().over(window_spec)) \
             .filter(col("row_num") == 1) \
             .drop("row_num")

# 2. Clean customer_id (handle NULL, ?, \N )
df_clean = df_clean.withColumn(
    "customer_id",
    when(col("customer_id").isNull(), "UNKNOWN")
    .when(col("customer_id").isin("NULL", "null", "\\N", "?"), "UNKNOWN")
    .otherwise(trim(col("customer_id")))
)

# 3. Standardize order_status (multilingual)
df_clean = df_clean.withColumn(
    "order_status",
    when(col("order_status").isin("completed", "已完成", "completado", "abgeschlossen", "पूर्ण"), "completed")
    .when(col("order_status").isin("pending", "pendiente", "待处理", "ausstehend"), "pending")
    .when(col("order_status").isin("cancelled", "cancelado", "storniert", "रद्द"), "cancelled")
    .when(col("order_status").isin("shipped", "भेज दिया"), "shipped")
    .when(col("order_status").isin("NULL", "null", "?", "\\N"), "unknown")
    .otherwise("unknown")
)

# 4. Standardize order_date (multiple formats)
df_clean = df_clean.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
        try_to_date(col("order_date"), "MM-dd-yyyy")
    )
)

#  fill null dates
df_clean = df_clean.withColumn(
    "order_date",
    coalesce(col("order_date"), lit("1900-01-01").cast("date"))
)

# 5. Fix numeric column
df_clean = df_clean.withColumn(
    "total_amount",
    col("total_amount").cast("double")
)

# 6. Clean other columns
df_clean = df_clean.withColumn("country_code", trim(col("country_code"))) \
                   .withColumn("channel", trim(col("channel"))) \
                   .withColumn("currency", trim(col("currency")))

# 7. Final selection
df_silver = df_clean.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_status",
    "country_code",
    "channel",
    "total_amount",
    "currency"
)

# 8. Write to silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novocart.silver.orders_cleaned")

# 9. Display
display(spark.table("novocart.silver.orders_cleaned"))

### ORDER ITEMS TABLE

In [0]:
from pyspark.sql.functions import col, expr, round, row_number, monotonically_increasing_id, when, trim, coalesce, lit
from pyspark.sql import Window

# Read from bronze
df = spark.table("novocart.bronze.order_items_raw")

# 1. Deduplicate
window_spec = Window.partitionBy("order_item_id").orderBy(monotonically_increasing_id())

df_clean = df.withColumn("row_num", row_number().over(window_spec)) \
             .filter(col("row_num") == 1) \
             .drop("row_num")

# 2. Clean product_id 
df_clean = df_clean.withColumn(
    "product_id",
    when(col("product_id").isNull(), "UNKNOWN_PRODUCT")
    .when(col("product_id").isin("NULL", "null", "\\N", "?"), "UNKNOWN_PRODUCT")
    .otherwise(trim(col("product_id")))
)

# 3. Handle quantity & unit_price
df_clean = df_clean.withColumn(
    "quantity",
    expr("try_cast(quantity as int)")
).withColumn(
    "unit_price",
    expr("try_cast(unit_price as double)")
)

# 4. Handle nulls after casting
df_clean = df_clean.withColumn("quantity", coalesce(col("quantity"), lit(0))) \
                   .withColumn("unit_price", coalesce(col("unit_price"), lit(0.0)))

# 5. Create line_total
df_clean = df_clean.withColumn(
    "line_total",
    round(col("quantity") * col("unit_price"), 2)
)

# 6. Select required columns
df_silver = df_clean.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total"
)

# 7. Write to silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novocart.silver.order_items_cleaned")

# 8. Display
display(spark.table("novocart.silver.order_items_cleaned"))

### PRODUCTS TABLE

In [0]:
from pyspark.sql.functions import col, when, trim, row_number, monotonically_increasing_id
from pyspark.sql import Window

# Read from bronze
df = spark.table("novocart.bronze.products_raw")

# 1. Remove duplicates  without changing row order
window_spec = Window.partitionBy("product_id").orderBy(monotonically_increasing_id())
df_clean = df.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

# 2. Clean product_name 
df_clean = df_clean.withColumn(
    "product_name",
    when(col("product_name").isNull(), "Unknown Product")
    .when(col("product_name") == "?", "Unknown Product")
    .otherwise(trim(col("product_name")))
)

# 3. Standardize text columns
df_clean = df_clean.withColumn("category", trim(col("category"))) \
                   .withColumn("currency", trim(col("currency"))) \
                   .withColumn("country_code", trim(col("country_code")))

# 4. Fix data types
df_clean = df_clean.withColumn("price", col("price").cast("double"))

# 5. filter invalid records
df_clean = df_clean.filter(col("product_id").isNotNull())

# 6. Final selection
df_silver = df_clean.select(
    "product_id",
    "product_name",
    "category",
    "price",
    "currency",
    "country_code"
)

# 7. Write to silver layer
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novocart.silver.products_cleaned")

display(spark.table("novocart.silver.products_cleaned"))

### EXCHANGE RATES TABLE

In [0]:
from pyspark.sql.functions import col, trim, row_number, monotonically_increasing_id, try_to_date
from pyspark.sql import Window

# Read from bronze
df = spark.table("novocart.bronze.exchange_rates_raw")

# 1. Deduplicate (based on currency_code + date)
window_spec = Window.partitionBy("currency_code", "effective_date") \
                    .orderBy(monotonically_increasing_id())

df_clean = df.withColumn("row_num", row_number().over(window_spec)) \
             .filter(col("row_num") == 1) \
             .drop("row_num")

# 2. Clean currency_code
df_clean = df_clean.withColumn(
    "currency_code",
    trim(col("currency_code"))
)

# 3. Fix data types
df_clean = df_clean.withColumn(
    "exchange_rate_to_usd",
    col("exchange_rate_to_usd").cast("double")
)

# 4. Standardize date
df_clean = df_clean.withColumn(
    "effective_date",
    try_to_date(col("effective_date"), "yyyy-MM-dd")
)

# 5. Final selection
df_silver = df_clean.select(
    "currency_code",
    "exchange_rate_to_usd",
    "effective_date"
)

# 6. Write to silver
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("novocart.silver.exchange_rates_cleaned")

# 7. Display
display(spark.table("novocart.silver.exchange_rates_cleaned"))